# Single-chain scope

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from types import SimpleNamespace

from cli import Cli
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset
from dataset_generator.types import DATASET_TYPE

# Define the absolute path for the root directory for outputs here (and if possible, make sure to have a prepared _dataset folder)
OUTPUTS_ROOT = Path("/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge")

# Create the SingleTransactionsDataset if it doesn't exist
dataset = SingleTransactionsDataset(root=str(OUTPUTS_ROOT / "_dataset"))
print(f"Dataset size: {len(dataset)} graphs")

/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Exporting data from database to CSV files... 
Dataset size: 1152 graphs


In [2]:
import os
import csv
import shutil


def split_single_transactions_dataset(
    source_root: str,
    train_root: str,
    test_root: str,
    test_tx_hashes,
    train_tx_hashes=None,
):
    source_processed = os.path.join(source_root, "processed")
    source_raw = os.path.join(source_root, "raw")

    with open(os.path.join(source_processed, "data_index.csv"), "r") as f:
        rows = list(csv.DictReader(f))

    test_tx_hashes = set(test_tx_hashes)
    missing = test_tx_hashes - {r["tx_hash"] for r in rows}
    if missing:
        raise ValueError(f"{len(missing)} train tx_hashes not found in source dataset: {sorted(missing)[:5]}...")

    test_rows = [r for r in rows if r["tx_hash"] in test_tx_hashes]

    if train_tx_hashes is None:
        train_rows = [r for r in rows if r["tx_hash"] not in test_tx_hashes]
    else:
        train_tx_hashes = set(train_tx_hashes)
        missing = train_tx_hashes - {r["tx_hash"] for r in rows}
        if missing:
            raise ValueError(f"{len(missing)} test tx_hashes not found in source dataset: {sorted(missing)[:5]}...")
        overlap = test_tx_hashes & train_tx_hashes
        if overlap:
            raise ValueError(f"{len(overlap)} tx_hashes appear in both splits: {sorted(overlap)[:5]}...")
        train_rows = [r for r in rows if r["tx_hash"] in train_tx_hashes]

    fieldnames = ["index", "graph_id", "bridge", "tx_hash", "file_name", "offset", "synthetic"]

    for split_root, split_rows in [(train_root, train_rows), (test_root, test_rows)]:
        split_raw = os.path.join(split_root, "raw")
        split_processed = os.path.join(split_root, "processed")
        os.makedirs(split_raw, exist_ok=True)
        os.makedirs(split_processed, exist_ok=True)

        # Copy raw CSVs unchanged so SingleTransactionsDataset.__init__ finds
        # raw_file_names present and doesn't trigger a DB download.
        for raw_file in os.listdir(source_raw):
            shutil.copy2(os.path.join(source_raw, raw_file), os.path.join(split_raw, raw_file))

        # Copy only the .pt files this split actually references (dedup, since
        # each file holds up to 50 graphs and several rows may share one file).
        referenced_files = {r["file_name"] for r in split_rows}
        for file_name in referenced_files:
            shutil.copy2(
                os.path.join(source_processed, file_name),
                os.path.join(split_processed, file_name),
            )

        # metadata.json (node/edge type vocab) is dataset-wide, safe to reuse as-is.
        shutil.copy2(
            os.path.join(source_processed, "metadata.json"),
            os.path.join(split_processed, "metadata.json"),
        )

        # Filtered index; file_name/offset are kept as-is since the .pt files were copied verbatim.
        with open(os.path.join(split_processed, "data_index.csv"), "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for new_idx, r in enumerate(split_rows):
                writer.writerow({**r, "index": new_idx})

    return len(train_rows), len(test_rows)


In [13]:
device = "cpu"  # Change to "cuda" if you have a GPU and want to use it
fixed_args = {
    "device": device,
    "dataset": DATASET_TYPE.SINGLE,
    "new_dataset": None,
    "force_reload": False,
    "num_workers": 0,

    "num_epochs": 100,
    "learning_rate": 0.005,
    "augment_factor": 0,
    "dme_threshold": 0.5,
    "rm_semantic_fusion": True,
    "early_stopping": 15,
    "kfolds": 5,
    "first_layer_channels": 128,
    "hidden_channels": 64,
    "dropout": 0.5,
    "input_drop": 0.0,
    "att_drop": 0.0,
    "n_fp_layers": 2,
    "n_mlp_layers": 2,
    "activation": "relu",
    "pooling": "mean",
    "residual": False,
    "random_seed": 42,
    "test_split": 0,    # Notice that we don't want train/test split on a training subset
    "rep_dim": 64,
    "eta": 1.0,
    "deep_sad_eps": 1.0,
    "grad_clip": 1.0,
    "weight_decay": 1e-6,
    "ae_pretrain": False,
}

inference_args = {
    "device": device,  # Change to "cuda" if you have a GPU
    "batch_size": 32,
    "num_workers": 0,
    "force_reload": False,
}

## Leak-free run against XChainWatcher labels

In [4]:
# Full-remove nomad-ronin1
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_xchainwatcher_leakfree")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_xchainwatcher_leakfree")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"ronin1", "nomad"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 
train: 420 graphs, test: 732 graphs


In [5]:
# Run leak-free training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="deepsad,comparison,xchainwatcher,leakfree"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 357 normal, 63 anomaly.
Identified 262 differential meta-paths for training.
Fold 1: centre initialised from 286 normal graphs.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 1.5853, val loss: 0.3983, PR-AUC (anomaly): 0.7284, f2-score (anomaly): 0.7746, tau: 1.0067e+00
  dist(normal) 3.8272e-01+-5.5448e-01 | dist(anomaly) 1.3641e+01+-1.5190e+01
              precision    recall  f1-score   support

           0       0.97      0.89      0.93        71
           1       0.58      0.85      0.69        13

    accuracy                           0.88        84
   macro avg       0.77      0.87      0.81        84
weighted avg       0.91      0.88      0.89        84

Fold 1/5, epoch 2/100 — training..

In [14]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="deepsad,comparison,xchainwatcher,leakfree"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_deepsad_comparison_xchainwatcher_leakfree__202608071212/models'.
Exporting data from database to CSV files... 
Loaded 732 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_xchainwatcher_leakfree'.
Fold 1: accuracy=0.7309 roc_auc=0.5303 pr_auc=0.2954 mcc=0.0480
Fold 2: accuracy=0.7855 roc_auc=0.7769 pr_auc=0.4130 mcc=0.3774
Fold 3: accuracy=0.6639 roc_auc=0.7420 pr_auc=0.3879 mcc=0.1447
Fold 4: accuracy=0.5014 roc_auc=0.5421 pr_auc=0.2453 mcc=-0.2743
Fold 5: accuracy=0.7691 roc_auc=0.8428 pr_auc=0.5541 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/sin

## 5050 testing against XChainWatcher labels

In [16]:
# 5050 testing (50% normal, 50% anomaly)
from collections import defaultdict

from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_xchainwatcher_5050")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_xchainwatcher_5050")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"ronin1", "nomad"}
bridge_to_tx_hashes_normal = defaultdict(list)
bridge_to_tx_hashes_anomaly = defaultdict(list)
for idx, entry in enumerate(source_ds.data_index):
    if entry["bridge"] in test_bridges:
        graph = source_ds[idx]
        if int(graph.y[0]) == 0:
            bridge_to_tx_hashes_normal[entry["bridge"]].append(entry["tx_hash"])
        else:
            bridge_to_tx_hashes_anomaly[entry["bridge"]].append(entry["tx_hash"])

test_tx_hashes = []
for bridge in test_bridges:
    tx_hashes = bridge_to_tx_hashes_normal[bridge]
    test_tx_hashes.extend(tx_hashes[: len(tx_hashes) // 2])
    tx_hashes = bridge_to_tx_hashes_anomaly[bridge]
    test_tx_hashes.extend(tx_hashes[: len(tx_hashes) // 2])

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")


Exporting data from database to CSV files... 
train: 787 graphs, test: 365 graphs


In [17]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="deepsad,comparison,xchainwatcher,5050"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 639 normal, 148 anomaly.
Identified 278 differential meta-paths for training.
Fold 1: centre initialised from 511 normal graphs.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 1.2881, val loss: 2.7540, PR-AUC (anomaly): 0.5627, f2-score (anomaly): 0.6030, tau: 1.7332e-01
  dist(normal) 3.3139e+00+-2.4169e+01 | dist(anomaly) 9.2667e+01+-1.3142e+02
              precision    recall  f1-score   support

           0       0.92      0.57      0.71       128
           1       0.30      0.80      0.44        30

    accuracy                           0.61       158
   macro avg       0.61      0.69      0.57       158
weighted avg       0.81      0.61      0.66       158

Fold 1/5, epoch 2/100 — training.

In [18]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="deepsad,comparison,xchainwatcher,5050"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_deepsad_comparison_xchainwatcher_5050__202608071221/models'.
Exporting data from database to CSV files... 
Loaded 365 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_xchainwatcher_5050'.
Fold 1: accuracy=0.8959 roc_auc=0.8681 pr_auc=0.7580 mcc=0.7329
Fold 2: accuracy=0.8548 roc_auc=0.9741 pr_auc=0.9352 mcc=0.7053
Fold 3: accuracy=0.8795 roc_auc=0.9423 pr_auc=0.8832 mcc=0.6347
Fold 4: accuracy=0.9178 roc_auc=0.9446 pr_auc=0.8207 mcc=0.7940
Fold 5: accuracy=0.9479 roc_auc=0.9905 pr_auc=0.9662 mcc=0.8537

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-inter

## Leak-free run against XScope labels

In [19]:
# Full-remove thorchain123-qubit
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_xscope_leakfree")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_xscope_leakfree")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"thorchain", "thorchain2", "thorchain3", "qubit"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 
train: 1107 graphs, test: 45 graphs


In [20]:
# Run leak-free training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="deepsad,comparison,xscope,leakfree"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 918 normal, 189 anomaly.
Identified 262 differential meta-paths for training.
Fold 1: centre initialised from 734 normal graphs.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 1.5497, val loss: 0.2089, PR-AUC (anomaly): 0.6912, f2-score (anomaly): 0.7547, tau: 1.4893e-01
  dist(normal) 1.3876e-01+-3.2821e-01 | dist(anomaly) 8.8742e+00+-1.2024e+01
              precision    recall  f1-score   support

           0       0.96      0.85      0.90       184
           1       0.53      0.84      0.65        38

    accuracy                           0.85       222
   macro avg       0.75      0.84      0.78       222
weighted avg       0.89      0.85      0.86       222

Fold 1/5, epoch 2/100 — training.

In [21]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="deepsad,comparison,xscope,leakfree"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_deepsad_comparison_xscope_leakfree__202608071226/models'.
Exporting data from database to CSV files... 
Loaded 45 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_xscope_leakfree'.
Fold 1: accuracy=0.5111 roc_auc=0.4884 pr_auc=0.9729 mcc=0.2017
Fold 2: accuracy=0.1778 roc_auc=0.4884 pr_auc=0.9729 mcc=0.0846
Fold 3: accuracy=0.6667 roc_auc=1.0000 pr_auc=1.0000 mcc=0.2768
Fold 4: accuracy=1.0000 roc_auc=1.0000 pr_auc=1.0000 mcc=1.0000
Fold 5: accuracy=0.1778 roc_auc=0.4884 pr_auc=0.9729 mcc=0.0846

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-e

## 5050 testing against XScope labels

In [22]:
# 5050 testing (50% normal, 50% anomaly)
from collections import defaultdict

from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_xscope_5050")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_xscope_5050")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XScope evaluation: Bridges thorchain, thorchain2, thorchain3, and qubit for testing, remaining cases for training.
test_bridges = {"thorchain", "thorchain2", "thorchain3", "qubit"}
bridge_to_tx_hashes_normal = defaultdict(list)
bridge_to_tx_hashes_anomaly = defaultdict(list)
for idx, entry in enumerate(source_ds.data_index):
    if entry["bridge"] in test_bridges:
        graph = source_ds[idx]
        if int(graph.y[0]) == 0:
            bridge_to_tx_hashes_normal[entry["bridge"]].append(entry["tx_hash"])
        else:
            bridge_to_tx_hashes_anomaly[entry["bridge"]].append(entry["tx_hash"])

test_tx_hashes = []
for bridge in test_bridges:
    tx_hashes = bridge_to_tx_hashes_normal[bridge]
    test_tx_hashes.extend(tx_hashes[: len(tx_hashes) // 2])
    tx_hashes = bridge_to_tx_hashes_anomaly[bridge]
    test_tx_hashes.extend(tx_hashes[: len(tx_hashes) // 2])

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")


Exporting data from database to CSV files... 
train: 1130 graphs, test: 22 graphs


In [23]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="deepsad,comparison,xscope,5050"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 919 normal, 211 anomaly.
Identified 261 differential meta-paths for training.
Fold 1: centre initialised from 736 normal graphs.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 1.2395, val loss: 0.5851, PR-AUC (anomaly): 0.5094, f2-score (anomaly): 0.6039, tau: 8.2038e-02
  dist(normal) 5.4536e-01+-5.2276e+00 | dist(anomaly) 4.0902e+01+-6.2596e+01
              precision    recall  f1-score   support

           0       1.00      0.23      0.37       183
           1       0.23      1.00      0.38        43

    accuracy                           0.38       226
   macro avg       0.62      0.61      0.38       226
weighted avg       0.85      0.38      0.37       226

Fold 1/5, epoch 2/100 — training.

In [24]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="deepsad,comparison,xscope,5050"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_deepsad_comparison_xscope_5050__202608071235/models'.
Exporting data from database to CSV files... 
Loaded 22 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_xscope_5050'.
Fold 1: accuracy=1.0000 roc_auc=1.0000 pr_auc=1.0000 mcc=1.0000
Fold 2: accuracy=0.6364 roc_auc=1.0000 pr_auc=1.0000 mcc=0.2623
Fold 3: accuracy=1.0000 roc_auc=1.0000 pr_auc=1.0000 mcc=1.0000
Fold 4: accuracy=1.0000 roc_auc=1.0000 pr_auc=1.0000 mcc=1.0000
Fold 5: accuracy=1.0000 roc_auc=1.0000 pr_auc=1.0000 mcc=1.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comp

## Middle-ground testing: test set with rotating incidents

### Incident: Nomad bridge

In [25]:
# Full-remove only nomad
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_nomadonly")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_nomadonly")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"nomad"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 
train: 422 graphs, test: 730 graphs


In [26]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="deepsad,comparison,nomadonly"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 357 normal, 65 anomaly.
Identified 261 differential meta-paths for training.
Fold 1: centre initialised from 285 normal graphs.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 1.5668, val loss: 0.4171, PR-AUC (anomaly): 0.6934, f2-score (anomaly): 0.7143, tau: 1.1576e+00
  dist(normal) 3.8148e-01+-5.5854e-01 | dist(anomaly) 1.3912e+01+-1.8921e+01
              precision    recall  f1-score   support

           0       0.96      0.89      0.92        72
           1       0.56      0.77      0.65        13

    accuracy                           0.87        85
   macro avg       0.76      0.83      0.78        85
weighted avg       0.89      0.87      0.88        85

Fold 1/5, epoch 2/100 — training..

In [27]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="deepsad,comparison,nomadonly"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_deepsad_comparison_nomadonly__202608071243/models'.
Exporting data from database to CSV files... 
Loaded 730 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_nomadonly'.
Fold 1: accuracy=0.6466 roc_auc=0.7431 pr_auc=0.4428 mcc=0.3813
Fold 2: accuracy=0.7219 roc_auc=0.6322 pr_auc=0.3400 mcc=-0.0205
Fold 3: accuracy=0.3329 roc_auc=0.4084 pr_auc=0.2750 mcc=-0.2248
Fold 4: accuracy=0.7370 roc_auc=0.7131 pr_auc=0.4027 mcc=0.0744
Fold 5: accuracy=0.5014 roc_auc=0.5127 pr_auc=0.3802 mcc=0.0068

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-compa

### Incident: Ronin #1 bridge

In [28]:
# Full-remove only ronin1
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_ronin1only")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_ronin1only")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and ronin1 for testing, remaining cases for training.
test_bridges = {"ronin1"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 
train: 1150 graphs, test: 2 graphs


In [29]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="deepsad,comparison,ronin1only"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 920 normal, 230 anomaly.
Identified 257 differential meta-paths for training.
Fold 1: centre initialised from 736 normal graphs.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 1.2120, val loss: 0.2200, PR-AUC (anomaly): 0.7798, f2-score (anomaly): 0.8537, tau: 2.2031e-01
  dist(normal) 1.1829e-01+-3.0175e-01 | dist(anomaly) 6.6285e+01+-1.3509e+02
              precision    recall  f1-score   support

           0       0.98      0.89      0.93       184
           1       0.68      0.91      0.78        46

    accuracy                           0.90       230
   macro avg       0.83      0.90      0.85       230
weighted avg       0.92      0.90      0.90       230

Fold 1/5, epoch 2/100 — training.

In [30]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="deepsad,comparison,ronin1only"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_deepsad_comparison_ronin1only__202608071247/models'.
Exporting data from database to CSV files... 
Loaded 2 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_ronin1only'.


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 1: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 2: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 3: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 4: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: U

Fold 5: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_deepsad_comparison_ronin1only__202608071247/inference_deepsad-comparison-ronin1only_202608071255


### Incident: THORchain #1 bridge

In [31]:
# Full-remove only thorchain1
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_thorchain1only")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_thorchain1only")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and thorchain1 for testing, remaining cases for training.
test_bridges = {"thorchain"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 
train: 1146 graphs, test: 6 graphs


In [32]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="deepsad,comparison,thorchain1only"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 920 normal, 226 anomaly.
Identified 257 differential meta-paths for training.
Fold 1: centre initialised from 736 normal graphs.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.9914, val loss: 2.0420, PR-AUC (anomaly): 0.6254, f2-score (anomaly): 0.6855, tau: 1.8736e-01
  dist(normal) 2.6266e+00+-2.5747e+01 | dist(anomaly) 1.4089e+02+-2.0654e+02
              precision    recall  f1-score   support

           0       0.93      0.84      0.88       184
           1       0.53      0.74      0.62        46

    accuracy                           0.82       230
   macro avg       0.73      0.79      0.75       230
weighted avg       0.85      0.82      0.83       230

Fold 1/5, epoch 2/100 — training.

In [33]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="deepsad,comparison,thorchain1only"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_deepsad_comparison_thorchain1only__202608071255/models'.
Exporting data from database to CSV files... 
Loaded 6 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_thorchain1only'.


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: U

Fold 1: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 2: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: U

Fold 3: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 4: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 5: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_deepsad_comparison_thorchain1only__202608071255/inference_deepsad-comparison-thorchain1only_202608071305


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


### Incident: THORchain #2 bridge

In [34]:
# Full-remove only thorchain2
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_thorchain2only")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_thorchain2only")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and thorchain2 for testing, remaining cases for training.
test_bridges = {"thorchain2"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 


train: 1137 graphs, test: 15 graphs


In [35]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="deepsad,comparison,thorchain2only"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 920 normal, 217 anomaly.
Identified 257 differential meta-paths for training.
Fold 1: centre initialised from 736 normal graphs.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 1.1527, val loss: 0.2206, PR-AUC (anomaly): 0.6981, f2-score (anomaly): 0.7738, tau: 6.1092e-02
  dist(normal) 7.4850e-02+-1.1223e-01 | dist(anomaly) 1.1923e+01+-1.8716e+01
              precision    recall  f1-score   support

           0       0.97      0.80      0.88       184
           1       0.51      0.89      0.65        44

    accuracy                           0.82       228
   macro avg       0.74      0.84      0.76       228
weighted avg       0.88      0.82      0.83       228

Fold 1/5, epoch 2/100 — training.

In [36]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="deepsad,comparison,thorchain2only"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_deepsad_comparison_thorchain2only__202608071305/models'.
Exporting data from database to CSV files... 
Loaded 15 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_thorchain2only'.


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 1: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 2: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 3: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 4: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 5: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_deepsad_comparison_thorchain2only__202608071305/inference_deepsad-comparison-thorchain2only_202608071312


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


### Incident: THORchain #3 bridge

In [37]:
# Full-remove only thorchain3
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_thorchain3only")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_thorchain3only")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and thorchain3 for testing, remaining cases for training.
test_bridges = {"thorchain3"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 
train: 1144 graphs, test: 8 graphs


In [38]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="deepsad,comparison,thorchain3only"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 918 normal, 226 anomaly.
Identified 257 differential meta-paths for training.
Fold 1: centre initialised from 735 normal graphs.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 2.0431, val loss: 1.1566, PR-AUC (anomaly): 0.3707, f2-score (anomaly): 0.4367, tau: 1.3932e-01
  dist(normal) 1.3419e+00+-8.1021e+00 | dist(anomaly) 1.3295e+01+-2.3071e+01
              precision    recall  f1-score   support

           0       0.86      0.86      0.86       183
           1       0.44      0.43      0.44        46

    accuracy                           0.78       229
   macro avg       0.65      0.65      0.65       229
weighted avg       0.78      0.78      0.78       229

Fold 1/5, epoch 2/100 — training.

In [39]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="deepsad,comparison,thorchain3only"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_deepsad_comparison_thorchain3only__202608071312/models'.
Exporting data from database to CSV files... 
Loaded 8 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_thorchain3only'.
Fold 1: accuracy=1.0000 roc_auc=1.0000 pr_auc=1.0000 mcc=1.0000
Fold 2: accuracy=1.0000 roc_auc=1.0000 pr_auc=1.0000 mcc=1.0000
Fold 3: accuracy=1.0000 roc_auc=1.0000 pr_auc=1.0000 mcc=1.0000
Fold 4: accuracy=1.0000 roc_auc=1.0000 pr_auc=1.0000 mcc=1.0000
Fold 5: accuracy=1.0000 roc_auc=1.0000 pr_auc=1.0000 mcc=1.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval

### Incident: Qubit bridge

In [40]:
# Full-remove only qubit
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_qubitonly")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_qubitonly")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and qubit for testing, remaining cases for training.
test_bridges = {"qubit"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 
train: 1136 graphs, test: 16 graphs


In [41]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="deepsad,comparison,qubitonly"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 920 normal, 216 anomaly.
Identified 257 differential meta-paths for training.
Fold 1: centre initialised from 736 normal graphs.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 1.4089, val loss: 0.3482, PR-AUC (anomaly): 0.7032, f2-score (anomaly): 0.7773, tau: 1.3508e-01
  dist(normal) 3.5298e-01+-2.2534e+00 | dist(anomaly) 3.1640e+01+-4.3624e+01
              precision    recall  f1-score   support

           0       0.96      0.86      0.91       184
           1       0.60      0.84      0.70        44

    accuracy                           0.86       228
   macro avg       0.78      0.85      0.80       228
weighted avg       0.89      0.86      0.87       228

Fold 1/5, epoch 2/100 — training.

In [42]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="deepsad,comparison,qubitonly"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_deepsad_comparison_qubitonly__202608071320/models'.
Exporting data from database to CSV files... 
Loaded 16 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_qubitonly'.


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 1: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 2: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 3: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 4: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 5: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_deepsad_comparison_qubitonly__202608071320/inference_deepsad-comparison-qubitonly_202608071328


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


In [43]:
# Full-remove only ronin1
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = "/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningdme-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset"  # existing generated dataset root
TRAIN_ROOT = "/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningdme-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_train_fullronin1"
TEST_ROOT = "/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/single-internaltx-eval-tuningdme-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_fullronin1"

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"ronin1"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 
train: 1150 graphs, test: 2 graphs


# Cross-chain scope

In [48]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from types import SimpleNamespace

from cli import Cli
from dataset_generator.cctx_dataset import CrossChainTransactionsDataset
from dataset_generator.types import DATASET_TYPE

# Define the absolute path for the root directory for outputs here (and if possible, make sure to have a prepared _dataset folder)
OUTPUTS_ROOT = Path("/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge")

# Create the CrossChainTransactionsDataset if it doesn't exist
dataset = CrossChainTransactionsDataset(root=str(OUTPUTS_ROOT / "_dataset"))
print(f"Dataset size: {len(dataset)} graphs")

Dataset size: 1241 graphs


In [49]:
import os
import csv
import shutil

# TODO CHANGE NAME
def split_single_transactions_dataset(
    source_root: str,
    train_root: str,
    test_root: str,
    test_tx_hashes,
    train_tx_hashes=None,
):
    source_processed = os.path.join(source_root, "processed")
    source_raw = os.path.join(source_root, "raw")

    with open(os.path.join(source_processed, "data_index.csv"), "r") as f:
        rows = list(csv.DictReader(f))

    test_tx_hashes = set(test_tx_hashes)
    missing = test_tx_hashes - {r["tx_hash"] for r in rows}
    if missing:
        raise ValueError(f"{len(missing)} train tx_hashes not found in source dataset: {sorted(missing)[:5]}...")

    test_rows = [r for r in rows if r["tx_hash"] in test_tx_hashes]

    if train_tx_hashes is None:
        train_rows = [r for r in rows if r["tx_hash"] not in test_tx_hashes]
    else:
        train_tx_hashes = set(train_tx_hashes)
        missing = train_tx_hashes - {r["tx_hash"] for r in rows}
        if missing:
            raise ValueError(f"{len(missing)} test tx_hashes not found in source dataset: {sorted(missing)[:5]}...")
        overlap = test_tx_hashes & train_tx_hashes
        if overlap:
            raise ValueError(f"{len(overlap)} tx_hashes appear in both splits: {sorted(overlap)[:5]}...")
        train_rows = [r for r in rows if r["tx_hash"] in train_tx_hashes]

    fieldnames = ["index", "cctx_graph_id", "tx_hash", "bridge", "file_name", "offset"]

    for split_root, split_rows in [(train_root, train_rows), (test_root, test_rows)]:
        split_raw = os.path.join(split_root, "raw")
        split_processed = os.path.join(split_root, "processed")
        os.makedirs(split_raw, exist_ok=True)
        os.makedirs(split_processed, exist_ok=True)

        # Copy raw CSVs unchanged so SingleTransactionsDataset.__init__ finds
        # raw_file_names present and doesn't trigger a DB download.
        for raw_file in os.listdir(source_raw):
            shutil.copy2(os.path.join(source_raw, raw_file), os.path.join(split_raw, raw_file))

        # Copy only the .pt files this split actually references (dedup, since
        # each file holds up to 50 graphs and several rows may share one file).
        referenced_files = {r["file_name"] for r in split_rows}
        for file_name in referenced_files:
            shutil.copy2(
                os.path.join(source_processed, file_name),
                os.path.join(split_processed, file_name),
            )

        # metadata.json (node/edge type vocab) is dataset-wide, safe to reuse as-is.
        shutil.copy2(
            os.path.join(source_processed, "metadata.json"),
            os.path.join(split_processed, "metadata.json"),
        )

        # Filtered index; file_name/offset are kept as-is since the .pt files were copied verbatim.
        with open(os.path.join(split_processed, "data_index.csv"), "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for new_idx, r in enumerate(split_rows):
                writer.writerow({**r, "index": new_idx})

    return len(train_rows), len(test_rows)


In [50]:
device = "cpu"  # Change to "cuda" if you have a GPU and want to use it
fixed_args = {
    "device": device,
    "dataset": DATASET_TYPE.CCTX,
    "new_dataset": None,
    "force_reload": False,
    "num_workers": 0,

    "num_epochs": 100,
    "learning_rate": 0.002,  # Notice that, for the new approach in cctx, we use a smaller learning rate than the default 0.005
    "augment_factor": 0,
    "dme_threshold": 0.5,
    "rm_semantic_fusion": True,
    "early_stopping": 15,
    "kfolds": 5,
    "first_layer_channels": 128,
    "hidden_channels": 64,
    "dropout": 0.5,
    "input_drop": 0.0,
    "att_drop": 0.0,
    "n_fp_layers": 2,
    "n_mlp_layers": 2,
    "activation": "relu",
    "pooling": "mean",
    "residual": False,
    "random_seed": 42,
    "test_split": 0,    # Notice that we don't want train/test split on a training subset
    "rep_dim": 64,
    "eta": 1.0,
    "deep_sad_eps": 1.0,
    "grad_clip": 1.0,
    "weight_decay": 1e-6,
    "ae_pretrain": False,
}

inference_args = {
    "device": device,  # Change to "cuda" if you have a GPU
    "batch_size": 32,
    "num_workers": 0,
    "force_reload": False,
}

## Leak-free run against XChainWatcher labels

In [52]:
# Full-remove nomad-ronin1
from dataset_generator.cctx_dataset import CrossChainTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_xchainwatcher_leakfree")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_xchainwatcher_leakfree")

source_ds = CrossChainTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"ronin1", "nomad"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

train: 1194 graphs, test: 47 graphs


In [53]:
# Run leak-free training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="deepsad,comparison,xchainwatcher,leakfree"
))

Early stopping: 15
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 1110 normal, 84 anomaly.
Identified 511 differential meta-paths for training.
Fold 1: centre initialised from 888 normal graphs.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.8614, val loss: 0.1245, PR-AUC (anomaly): 0.2481, f2-score (anomaly): 0.5782, tau: 3.2458e-02
  dist(normal) 4.5602e-02+-7.1391e-02 | dist(anomaly) 7.4737e-02+-1.4336e-01
              precision    recall  f1-score   support

           0       1.00      0.72      0.84       222
           1       0.22      1.00      0.35        17

    accuracy                           0.74       239
   macro avg       0.61      0.86      0.60       239
weighted avg       0.94      0.74      0.80       239

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.2389, val 

In [54]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="deepsad,comparison,xchainwatcher,leakfree"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.002_nosf_es15_ts0_deepsad_comparison_xchainwatcher_leakfree__202608071339/models'.
Loaded 47 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_xchainwatcher_leakfree'.


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Fold 1: accuracy=1.0000 roc_auc=nan pr_auc=0.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Fold 2: accuracy=0.4255 roc_auc=nan pr_auc=0.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Fold 3: accuracy=1.0000 roc_auc=nan pr_auc=0.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Fold 4: accuracy=1.0000 roc_auc=nan pr_auc=0.0000 mcc=0.0000
Fold 5: accuracy=0.9787 roc_auc=nan pr_auc=0.0000 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.002_nosf_es15_ts0_deepsad_comparison_xchainwatcher_leakfree__202608071339/inference_deepsad-comparison-xchainwatcher-leakfree_202608071355


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


## 5050 testing against XChainWatcher labels

In [55]:
# 5050 testing (50% normal, 50% anomaly)
from collections import defaultdict

from dataset_generator.cctx_dataset import CrossChainTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_xchainwatcher_5050")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_xchainwatcher_5050")

source_ds = CrossChainTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"ronin1", "nomad"}
bridge_to_tx_hashes_normal = defaultdict(list)
bridge_to_tx_hashes_anomaly = defaultdict(list)
for idx, entry in enumerate(source_ds.data_index):
    if entry["bridge"] in test_bridges:
        graph = source_ds[idx]
        if int(graph.y[0]) == 0:
            bridge_to_tx_hashes_normal[entry["bridge"]].append(entry["tx_hash"])
        else:
            bridge_to_tx_hashes_anomaly[entry["bridge"]].append(entry["tx_hash"])

test_tx_hashes = []
for bridge in test_bridges:
    tx_hashes = bridge_to_tx_hashes_normal[bridge]
    test_tx_hashes.extend(tx_hashes[: len(tx_hashes) // 2])
    tx_hashes = bridge_to_tx_hashes_anomaly[bridge]
    test_tx_hashes.extend(tx_hashes[: len(tx_hashes) // 2])

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")


train: 1218 graphs, test: 23 graphs


In [56]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="deepsad,comparison,xchainwatcher,5050"
))

Early stopping: 15


--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 1134 normal, 84 anomaly.
Identified 496 differential meta-paths for training.
Fold 1: centre initialised from 907 normal graphs.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.7852, val loss: 0.1556, PR-AUC (anomaly): 0.0517, f2-score (anomaly): 0.2852, tau: 3.0173e-02
  dist(normal) 9.2691e-02+-1.4287e-01 | dist(anomaly) 5.3244e-02+-1.6401e-02
              precision    recall  f1-score   support

           0       1.00      0.06      0.12       227
           1       0.07      1.00      0.14        17

    accuracy                           0.13       244
   macro avg       0.54      0.53      0.13       244
weighted avg       0.94      0.13      0.12       244

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.2265, val loss: 0.1115, PR-AU

In [57]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="deepsad,comparison,xchainwatcher,5050"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.002_nosf_es15_ts0_deepsad_comparison_xchainwatcher_5050__202608071356/models'.
Loaded 23 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_xchainwatcher_5050'.


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Fold 1: accuracy=0.1739 roc_auc=nan pr_auc=0.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Fold 2: accuracy=1.0000 roc_auc=nan pr_auc=0.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Fold 3: accuracy=1.0000 roc_auc=nan pr_auc=0.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Fold 4: accuracy=1.0000 roc_auc=nan pr_auc=0.0000 mcc=0.0000
Fold 5: accuracy=1.0000 roc_auc=nan pr_auc=0.0000 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.002_nosf_es15_ts0_deepsad_comparison_xchainwatcher_5050__202608071356/inference_deepsad-comparison-xchainwatcher-5050_202608071413


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


## Leak-free run against XScope labels

In [58]:
# Full-remove thorchain123-qubit
from dataset_generator.cctx_dataset import CrossChainTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_xscope_leakfree")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_xscope_leakfree")

source_ds = CrossChainTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"thorchain", "thorchain2", "thorchain3", "qubit"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

train: 1206 graphs, test: 35 graphs


In [59]:
# Run leak-free training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="deepsad,comparison,xscope,leakfree"
))

Early stopping: 15
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 1146 normal, 60 anomaly.
Identified 499 differential meta-paths for training.
Fold 1: centre initialised from 916 normal graphs.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.9392, val loss: 0.3447, PR-AUC (anomaly): 0.1359, f2-score (anomaly): 0.4583, tau: 2.7316e-01
  dist(normal) 3.3225e-01+-5.9110e-01 | dist(anomaly) 3.0701e-01+-7.5636e-02
              precision    recall  f1-score   support

           0       0.99      0.73      0.84       230
           1       0.15      0.92      0.26        12

    accuracy                           0.74       242
   macro avg       0.57      0.83      0.55       242
weighted avg       0.95      0.74      0.82       242

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.2716, val 

In [60]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="deepsad,comparison,xscope,leakfree"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.002_nosf_es15_ts0_deepsad_comparison_xscope_leakfree__202608071414/models'.
Loaded 35 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_xscope_leakfree'.
Fold 1: accuracy=0.5429 roc_auc=0.1818 pr_auc=0.5434 mcc=-0.0606
Fold 2: accuracy=0.3143 roc_auc=0.2424 pr_auc=0.5815 mcc=0.0000
Fold 3: accuracy=0.2286 roc_auc=0.0000 pr_auc=0.4929 mcc=-0.4523
Fold 4: accuracy=0.4000 roc_auc=0.2386 pr_auc=0.6640 mcc=-0.0241
Fold 5: accuracy=0.2286 roc_auc=0.0000 pr_auc=0.4992 mcc=-0.4523

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/

## 5050 testing against XScope labels

In [61]:
# 5050 testing (50% normal, 50% anomaly)
from collections import defaultdict

from dataset_generator.cctx_dataset import CrossChainTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_xscope_5050")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_xscope_5050")

source_ds = CrossChainTransactionsDataset(root=SOURCE_ROOT)

# XScope evaluation: Bridges thorchain, thorchain2, thorchain3, and qubit for testing, remaining cases for training.
test_bridges = {"thorchain", "thorchain2", "thorchain3", "qubit"}
bridge_to_tx_hashes_normal = defaultdict(list)
bridge_to_tx_hashes_anomaly = defaultdict(list)
for idx, entry in enumerate(source_ds.data_index):
    if entry["bridge"] in test_bridges:
        graph = source_ds[idx]
        if int(graph.y[0]) == 0:
            bridge_to_tx_hashes_normal[entry["bridge"]].append(entry["tx_hash"])
        else:
            bridge_to_tx_hashes_anomaly[entry["bridge"]].append(entry["tx_hash"])

test_tx_hashes = []
for bridge in test_bridges:
    tx_hashes = bridge_to_tx_hashes_normal[bridge]
    test_tx_hashes.extend(tx_hashes[: len(tx_hashes) // 2])
    tx_hashes = bridge_to_tx_hashes_anomaly[bridge]
    test_tx_hashes.extend(tx_hashes[: len(tx_hashes) // 2])

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")


train: 1224 graphs, test: 17 graphs


In [62]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="deepsad,comparison,xscope,5050"
))

Early stopping: 15
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 1152 normal, 72 anomaly.
Identified 493 differential meta-paths for training.
Fold 1: centre initialised from 921 normal graphs.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.7923, val loss: 0.1127, PR-AUC (anomaly): 0.0543, f2-score (anomaly): 0.2915, tau: 3.0612e-02
  dist(normal) 6.1241e-02+-7.4037e-02 | dist(anomaly) 4.5407e-02+-3.9319e-02
              precision    recall  f1-score   support

           0       0.99      0.33      0.50       231
           1       0.08      0.93      0.14        14

    accuracy                           0.37       245
   macro avg       0.53      0.63      0.32       245
weighted avg       0.94      0.37      0.48       245

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.1964, val 

In [ ]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="deepsad,comparison,xscope,5050"
))

## Middle-ground testing: test set with rotating incidents

### Incident: THORchain #2 bridge

In [74]:
# Full-remove only thorchain2
from dataset_generator.cctx_dataset import CrossChainTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_thorchain2only")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_thorchain2only")

source_ds = CrossChainTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and thorchain2 for testing, remaining cases for training.
test_bridges = {"thorchain2"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

train: 1239 graphs, test: 2 graphs


In [75]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="deepsad,comparison,thorchain2only"
))

Early stopping: 15
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 1157 normal, 82 anomaly.
Identified 489 differential meta-paths for training.
Fold 1: centre initialised from 925 normal graphs.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.7621, val loss: 0.1238, PR-AUC (anomaly): 0.1250, f2-score (anomaly): 0.2685, tau: 3.5430e-02
  dist(normal) 6.7031e-02+-3.5815e-02 | dist(anomaly) 9.2914e-02+-1.4908e-01
              precision    recall  f1-score   support

           0       1.00      0.06      0.11       232
           1       0.07      1.00      0.13        16

    accuracy                           0.12       248
   macro avg       0.53      0.53      0.12       248
weighted avg       0.94      0.12      0.11       248

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.1892, val 

In [76]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="deepsad,comparison,thorchain2only"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.002_nosf_es15_ts0_deepsad_comparison_thorchain2only__202608071712/models'.
Loaded 2 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_thorchain2only'.


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 1: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 2: accuracy=0.5000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 3: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 4: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 5: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.002_nosf_es15_ts0_deepsad_comparison_thorchain2only__202608071712/inference_deepsad-comparison-thorchain2only_202608071730


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


### Incident: THORchain #3 bridge

In [77]:
# Full-remove only thorchain3
from dataset_generator.cctx_dataset import CrossChainTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_thorchain3only")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_thorchain3only")

source_ds = CrossChainTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and thorchain3 for testing, remaining cases for training.
test_bridges = {"thorchain3"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

train: 1235 graphs, test: 6 graphs


In [78]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="deepsad,comparison,thorchain3only"
))

Early stopping: 15
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 1157 normal, 78 anomaly.
Identified 488 differential meta-paths for training.
Fold 1: centre initialised from 925 normal graphs.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.7806, val loss: 0.1943, PR-AUC (anomaly): 0.0610, f2-score (anomaly): 0.3571, tau: 3.9041e-02
  dist(normal) 1.4691e-01+-3.9098e-01 | dist(anomaly) 5.0605e-02+-2.3252e-02
              precision    recall  f1-score   support

           0       1.00      0.42      0.59       232
           1       0.10      1.00      0.18        15

    accuracy                           0.45       247
   macro avg       0.55      0.71      0.39       247
weighted avg       0.95      0.45      0.56       247

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.2261, val 

In [79]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="deepsad,comparison,thorchain3only"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.002_nosf_es15_ts0_deepsad_comparison_thorchain3only__202608071731/models'.
Loaded 6 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_thorchain3only'.


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 1: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 2: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 3: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 4: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 5: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.002_nosf_es15_ts0_deepsad_comparison_thorchain3only__202608071731/inference_deepsad-comparison-thorchain3only_202608071753


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


### Incident: Qubit bridge

In [80]:
# Full-remove only qubit
from dataset_generator.cctx_dataset import CrossChainTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_qubitonly")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_qubitonly")

source_ds = CrossChainTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and qubit for testing, remaining cases for training.
test_bridges = {"qubit"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

train: 1214 graphs, test: 27 graphs


In [81]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="deepsad,comparison,qubitonly"
))

Early stopping: 15
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 1146 normal, 68 anomaly.
Identified 495 differential meta-paths for training.
Fold 1: centre initialised from 916 normal graphs.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.6677, val loss: 0.0966, PR-AUC (anomaly): 0.1434, f2-score (anomaly): 0.3333, tau: 2.6771e-02
  dist(normal) 5.3795e-02+-8.8603e-02 | dist(anomaly) 8.5246e-02+-1.6709e-01
              precision    recall  f1-score   support

           0       0.98      0.56      0.71       230
           1       0.10      0.85      0.17        13

    accuracy                           0.57       243
   macro avg       0.54      0.70      0.44       243
weighted avg       0.94      0.57      0.68       243

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.2084, val 

In [82]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="deepsad,comparison,qubitonly"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.002_nosf_es15_ts0_deepsad_comparison_qubitonly__202608071754/models'.
Loaded 27 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_qubitonly'.
Fold 1: accuracy=0.8889 roc_auc=0.7273 pr_auc=0.6955 mcc=0.7826
Fold 2: accuracy=0.2963 roc_auc=0.0000 pr_auc=0.4129 mcc=-0.4264
Fold 3: accuracy=0.2963 roc_auc=0.0057 pr_auc=0.4117 mcc=-0.4264
Fold 4: accuracy=0.2963 roc_auc=0.7273 pr_auc=0.6916 mcc=-0.4264
Fold 5: accuracy=0.4074 roc_auc=0.1023 pr_auc=0.4404 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs1